<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现字节对编码（BPE）分词器 — 简化版

- 本 notebook 从零实现流行的字节对编码（BPE）分词算法，用于教学；GPT-2 至 GPT-4、Llama 3 等模型均使用 BPE
- 分词目的详见 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；此处为解释 BPE 算法的 bonus 材料
- OpenAI 为原始 GPT 模型实现的 BPE 分词器见 [此处](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- BPE 算法最初由 Philip Gage 于 1994 年提出："[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 如今包括 Llama 3 在内的大多数项目使用 OpenAI 开源 [tiktoken](https://github.com/openai/tiktoken)，因其计算性能高；可加载预训练 GPT-2/GPT-4 分词器（Llama 3 也使用 GPT-4 分词器训练）
- 与上述实现不同，本 notebook 的实现还包含训练分词器的函数（教学用途）
- 另有 [minBPE](https://github.com/karpathy/minbpe) 支持训练，可能更高效（本实现侧重教学）；相比 `minbpe`，本实现还可加载原始 OpenAI 词表与 merges

**这是非常简化的教学实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) 中有更完善（但更难读）的实现，行为与 tiktoken 一致。**

&nbsp;
# 1. 字节对编码（BPE）的核心思想

- BPE 的核心思想是将文本转为整数表示（token ID）供 LLM 训练（见 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 位与字节

- 在介绍 BPE 算法前，先说明字节（byte）的概念
- 考虑将文本转为字节数组（BPE 即 "byte" pair encoding）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 对 `bytearray` 调用 `list()` 时，每个字节作为独立元素，结果为对应字节值的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这可将文本转为 LLM 嵌入层所需的 token ID 表示
- 但缺点是每个字符一个 ID（短文本也会产生很多 ID！）
- 即 17 字符输入需 17 个 token ID 作为 LLM 输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 若接触过 LLM，可知 BPE 分词器词表包含整词或子词的 token ID，而非每字符一个
- 例如 GPT-2 将 "This is some text" 分词为 4 个而非 17 个 token：`1212, 318, 617, 2420`
- 可用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 验证：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- 一个字节 8 位，共 2<sup>8</sup> = 256 种取值（0–255）
- 执行 `bytearray(range(0, 257))` 会警告 `ValueError: byte must be in range(0, 256)`
- BPE 分词器通常将这 256 值作为前 256 个单字符 token；可用以下代码查看：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 注意 256、257 项不是单字符而是双字符（空格+字母），这是原始 GPT-2 BPE 的小缺陷（GPT-4 分词器已改进）

&nbsp;
## 1.2 构建词表

- BPE 分词算法目标是构建常见子词词表，如 `298: ent`（出现在 *entangle, entertain, enter, entrance, entity, ...* 等词中），甚至完整词如 

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最初由 Philip Gage 于 1994 年提出："[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 在进入代码实现前，先总结当今 LLM 分词器使用的 BPE 形式：

&nbsp;
## 1.3 BPE 算法概要

**1. 识别高频对**
- 每轮扫描文本，找出最常见的字节（或字符）对

**2. 替换并记录**

- 将该对替换为新占位 ID（尚未使用的 ID，如从 0…255 开始则首个为 256）
- 在查找表中记录映射
- 查找表大小为超参数，即「词表大小」（GPT-2 为 50,257）

**3. 重复直至无收益**

- 重复步骤 1、2，持续合并最高频对
- 无法进一步压缩时停止（如每对最多出现一次）

**解压缩（解码）**

- 用查找表将每个 ID 替换回对应字符对，逆序还原原文



&nbsp;
## 1.4 BPE 算法示例

### 1.4.1 编码部分的具体示例（1.3 节步骤 1 和 2）

- 假设训练文本为 `the cat in the hat`，要为其构建 BPE 词表

**第 1 轮迭代**

1. 识别高频对
  - 文本中 "th" 出现两次（开头及第二个 "e" 前）

2. 替换并记录
  - 将 "th" 替换为未使用的 token ID，如 256
  - 新文本：`<256>e cat in <256>e hat`
  - 新词表：

```
  0: ...
  ...
  256: "th"
```

**第 2 轮迭代**

1. **识别高频对**  
   - 在 `<256>e cat in <256>e hat` 中，`<256>e` 出现两次

2. **替换并记录**  
   - 将 `<256>e` 替换为新 ID，如 `257`  
   - 新文本：
     ```
     <257> cat in <257> hat
     ```
   - 更新词表：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**第 3 轮迭代**

1. **识别高频对**  
   - 在 `<257> cat in <257> hat` 中，`<257> ` 出现两次（开头及 “hat” 前）

2. **替换并记录**  
   - 将 `<257> ` 替换为新 ID，如 `258`  
   - 新文本：
     ```
     <258>cat in <258>hat
     ```
   - 更新词表：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 依此类推

&nbsp;
### 1.4.2 解码部分的具体示例（1.3 节步骤 3）

- 还原原文：按引入顺序的逆序，将每个 token ID 替换回对应字符对
- 从最终压缩文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`  
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 简易 BPE 实现

- 下面将实现上述算法的 Python 类，接口类似 `tiktoken`
- 上文编码部分描述 `train()` 的训练步骤；`encode()` 类似（因特殊 token 处理而更复杂）：

1. 将输入文本拆分为单个字节
2. 反复查找并合并相邻 token 对（与已学 BPE merges 匹配，按 rank 从高到低，即学习顺序）
3. 直至无法继续合并
4. 最终 token ID 列表即为编码结果

In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Now create the vocab and inverse vocab dictionaries
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with a space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### 短 token 序列的边界情况处理

BPE 合并需要相邻 token 对。
若 token 序列少于 2 项，则不存在 token 对，`find_freq_pair` 返回 `None`，训练优雅停止。

In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- 上述 `BPETokenizerSimple` 类代码较多，本 notebook 不逐行讨论，下一节简要介绍用法以理解类方法

## 3. BPE 实现走读

- 实践中强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)；本实现侧重可读性与教学，非性能
- 用法与 tiktoken 大致相似，但 tiktoken 无训练方法
- 下面通过示例了解 `BPETokenizerSimple` 如何工作（不展开代码细节）

### 3.1 训练、编码与解码

- 首先用一些样本文本作为训练数据集：

In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- 接下来初始化并训练词表大小为 1,000 的 BPE 分词器
- 词表默认已有 255 项（字节值），实际学习 745 项
- 对比：GPT-2 词表 50,257；GPT-4 为 100,256（`cl100k_base`）；GPT-4o 为 199,997（`o200k_base`）；它们训练集远大于我们的示例文本

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 可查看词表内容（注意会输出很长列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 该词表通过约 742 次合并创建（~ `1000 - len(range(0, 256))`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 即前 256 项为单字符 token

- 接下来用 `encode` 方法及已学习的 merges 编码文本：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- 从长度可见，42 字符句子编码为 20 个 token ID，相比逐字符编码约减半

- `decode()` 使用词表将 token ID 映射回文本：

In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- 逐 token ID 迭代可更好理解解码过程：

In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- 可见多数 token ID 表示 2 字符子词；因训练文本短、重复词少，且词表较小

- 总结：`decode(encode())` 应能还原任意输入文本：

In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. 总结

- 以上就是 BPE 的要点，包含创建新分词器的训练方法
- 希望本教程对教学有帮助；有问题欢迎在 [此处](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a) 发起讨论


**这是非常简化的教学实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) 中有更完善（但更难读）的实现，行为与 tiktoken 一致。**